# Stage 1: SKAB Dataset Exploration & Audit
**Motor Predictive Maintenance - IoT Induction Motor Project**

In this notebook, we audit all 35 CSV files in the SKAB benchmark and inspect sensor feature distributions.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, "../ml")
from dataset_audit import audit_dataset

data_path = Path("../data/raw/SKAB")
SELECTED_FEATURES = ["Accelerometer1RMS", "Accelerometer2RMS", "Current", "Temperature", "Thermocouple", "Voltage"]
print("Data path:", data_path.resolve())


Data path: C:\Users\himes\Desktop\Cornerstone\data\raw\SKAB


## 1. Run Comprehensive Audit Across All 35 Files

In [2]:
audit_csv = Path("../data/processed/skab_dataset_audit.csv")
audit_df = audit_dataset(data_path, audit_csv)
print(f"Total files audited: {len(audit_df)}")

# Summary by folder
summary = audit_df.groupby("folder").agg(
    file_count=("filename", "count"),
    total_rows=("rows", "sum"),
    nulls=("nulls", "sum"),
    mean_anomaly_pct=("anomaly_ratio_pct", "mean"),
    total_changepoints=("changepoints", "sum")
)
display(summary)


Found 4 folders in ..\data\raw\SKAB
Saved audit summary to: ..\data\processed\skab_dataset_audit.csv
Total files audited: 35


,file_count,total_rows,nulls,mean_anomaly_pct,total_changepoints
folder,,,,,
anomaly-free,1,9401,0,0.000000,0
other,14,14985,0,36.407143,51
valve1,16,18162,0,34.723125,63
valve2,4,4312,0,35.260000,16


## 2. Sensor Value Distribution: Normal vs Anomaly Recording

In [3]:
df_norm = pd.read_csv(data_path / "anomaly-free/anomaly-free.csv", sep=";")
df_anomaly = pd.read_csv(data_path / "valve1/0.csv", sep=";")

print("--- Normal Operation Feature Summary (anomaly-free.csv) ---")
display(df_norm[SELECTED_FEATURES].describe().round(3).T)

print("--- Valve 1 Anomaly Recording Feature Summary (valve1/0.csv) ---")
display(df_anomaly[SELECTED_FEATURES].describe().round(3).T)


--- Normal Operation Feature Summary (anomaly-free.csv) ---


,count,mean,std,min,25%,50%,75%,max
Accelerometer1RMS,9401.0,0.212,0.005,0.194,0.210,0.213,0.215,0.226
Accelerometer2RMS,9401.0,0.269,0.004,0.248,0.266,0.267,0.272,0.280
Current,9401.0,2.568,6.082,0.855,2.130,2.541,2.728,233.203
Temperature,9401.0,89.507,0.673,88.171,88.974,89.380,89.922,91.725
Thermocouple,9401.0,28.418,0.758,26.766,27.842,28.625,29.068,29.522
Voltage,9401.0,228.491,12.580,1.037,222.625,228.942,234.963,252.806


--- Valve 1 Anomaly Recording Feature Summary (valve1/0.csv) ---


,count,mean,std,min,25%,50%,75%,max
Accelerometer1RMS,1148.0,0.027,0.000,0.026,0.026,0.027,0.027,0.027
Accelerometer2RMS,1148.0,0.040,0.001,0.038,0.040,0.040,0.041,0.043
Current,1148.0,1.410,9.712,0.388,0.794,1.035,1.223,235.511
Temperature,1148.0,77.437,1.745,74.237,75.663,78.384,78.917,79.889
Thermocouple,1148.0,25.958,0.078,25.830,25.879,25.955,26.020,26.104
Voltage,1148.0,230.322,15.910,1.087,225.266,231.316,237.097,255.324


## 3. Visualize Normal vs Anomaly Time Series

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=False)

ax1.plot(pd.to_datetime(df_norm["datetime"][:1000]), df_norm["Current"][:1000], color="blue", label="Normal Current (A)")
ax1.set_title("Healthy Baseline Operation Excerpt (anomaly-free.csv)")
ax1.set_ylabel("Current (A)")
ax1.grid(True, linestyle="--", alpha=0.5)

ax2.plot(pd.to_datetime(df_anomaly["datetime"]), df_anomaly["Current"], color="darkred", label="Current (A)")
anom_pts = df_anomaly.loc[df_anomaly["anomaly"] == 1.0, "datetime"]
if len(anom_pts) > 0:
    ax2.axvspan(pd.to_datetime(anom_pts.iloc[0]), pd.to_datetime(anom_pts.iloc[-1]), color="salmon", alpha=0.35, label="Ground Truth Anomaly Window")
ax2.set_title("Held-out Experiment with Anomaly Window (valve1/0.csv)")
ax2.set_ylabel("Current (A)")
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()
